# Batch 8 — Network-timeout / performance cluster

Covers: **M1-21, M1-22, M1-23, M1-24, M2-3**

A themed cluster: the OTP-submit connection-error messaging gap (M1-21/M1-22, same call site), the one
outbound call that used to have no `timeout` at all (M1-23, confirm-only -- already fixed by earlier
retry-logic work), the ABHA card/QR download's per-chunk-vs-total timeout gap (M1-24), and consent storage
lookup performance at scale (M2-3).

Every check below runs against the REAL repo code (`server/`, `tools/`), isolated via `harness.py`'s scratch
storage (json_file_store checks) or direct `unittest.mock.patch`/`monkeypatch`-style stand-ins for
`requests.get`/`requests.post` (network checks) -- nothing here touches the real `storage/`/`logs/`
directories or calls real ABDM.

In [ ]:
import sys, os, time, json as _json
from pathlib import Path

# Locate the repo root by walking up from wherever `jupyter execute` sets
# the working directory (its default is the notebook's OWN directory,
# tools/edge_case_testing/notebooks -- not the repo root) until we find a
# directory containing both server/ and tools/.
_here = Path.cwd()
REPO_ROOT = None
for _candidate in [_here, *_here.parents]:
    if (_candidate / "server").is_dir() and (_candidate / "tools").is_dir():
        REPO_ROOT = _candidate
        break
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root by walking up from cwd={_here}")

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "tools" / "edge_case_testing" / "notebooks"))
os.chdir(REPO_ROOT)

import inspect
import tempfile
from unittest.mock import patch, MagicMock

import requests

import harness
from harness import FakeResponse, CallRecorder, check

harness.activate_scratch_storage('batch8')


## M1-23 — `update_bridge_url()` has a `timeout` (confirm-only)

The test plan flags `server/auth.py`'s `update_bridge_url()` as "the ONE outbound call in the entire
codebase with no `timeout` parameter at all". Reading the current source shows this was already fixed by
earlier retry-logic work -- its docstring says so explicitly. This cell confirms it directly from the
live source (not from the docstring's claim alone), by inspecting the actual call for a `timeout=` kwarg.

In [ ]:
import server.auth as auth_module

src = inspect.getsource(auth_module.update_bridge_url)
check("M1-23: update_bridge_url() source contains a timeout= kwarg on its outbound call",
      "timeout=30" in src or "timeout =" in src)
print(src)


## M1-21 / M1-22 — connection-error messaging during Aadhaar enrollment OTP submit

Both cases are about the SAME call site: `enroll_by_aadhaar()` inside
`tools/m1_test_suite/flows/enrollment.py`'s `run()`. M1-21 is a full network drop right after typing the
OTP (a `ConnectionError` before submit even reaches the network); M1-22 is a proxy that withholds the
response past the client timeout while ABDM actually completes the enrollment server-side (a `Timeout`
after `call_with_retry` exhausts). Before the fix, a `RequestException` from either scenario propagated
uncaught out of `run()` to `cli.py`'s generic top-level catch-all, which prints only
`"Flow 1: ABHA Enrollment via Aadhaar raised an unexpected error: ConnectionError: ..."` -- zero statement
about whether the enrollment might have actually gone through, and (per M1-22's own text) a tester would
then naturally re-run the flow, hit ABDM's "already exists" response, and see a calm
"this is a normal success outcome" message that masks which attempt actually created the account.

This exercises `enrollment.run()` directly with `request_otp`/`enroll_by_aadhaar` patched to raise
`requests.exceptions.ConnectionError`/`requests.exceptions.Timeout`, and `input()` patched to feed the
prompts, checking that `run()` returns cleanly (no uncaught exception) and that the honest
"connection-level failure ... not known from here ... do not assume it's safe to just retry" message is
printed via `report_connection_error()`.

In [ ]:
import io
import contextlib

import tools.m1_test_suite.flows.enrollment as enrollment_module


def run_enrollment_with_stub_exception(exc, patch_target):
    """Runs enrollment.run() with `patch_target` (either 'request_otp' or
    'enroll_by_aadhaar') raising `exc`, and canned input() answers for the
    two prompts. Returns (result, printed_output)."""
    answers = iter(["999999999999", "9876543210", "123456"])

    def fake_input(_prompt=""):
        return next(answers)

    buf = io.StringIO()
    with patch.object(enrollment_module, "request_otp") as mock_otp, \
         patch.object(enrollment_module, "enroll_by_aadhaar") as mock_enroll, \
         patch.object(enrollment_module, "encrypt", return_value="encrypted-stub-value"), \
         patch("builtins.input", fake_input), \
         contextlib.redirect_stdout(buf):
        # encrypt() is also patched above -- it's a thin wrapper around
        # server/crypto.py's get_public_certificate()/encrypt_value(),
        # and get_public_certificate() does a REAL network fetch of
        # ABDM's public cert (see server/crypto.py). Not mocking it would
        # make this cell's "connection error" scenario accidentally test
        # a totally different, unrelated real network call instead of
        # the request_otp()/enroll_by_aadhaar() call this cell actually
        # targets.

        if patch_target == "request_otp":
            mock_otp.side_effect = exc
        else:
            mock_otp.return_value = FakeResponse(200, {"txnId": "txn-123", "message": "OTP sent"})
            mock_enroll.side_effect = exc

        result = enrollment_module.run()

    return result, buf.getvalue()


# M1-21: full network drop -- ConnectionError right at OTP submit (enroll_by_aadhaar)
result_21, output_21 = run_enrollment_with_stub_exception(
    requests.exceptions.ConnectionError("Failed to establish a new connection: [Errno 111] Connection refused"),
    "enroll_by_aadhaar",
)
check("M1-21: run() returns cleanly instead of raising (no uncaught exception reaches cli.py)", isinstance(result_21, dict))
check("M1-21: honest 'connection-level failure' message printed", "connection-level failure" in output_21)
check("M1-21: explicitly says outcome is not known from here", "known from here" in output_21.lower())
check("M1-21: warns against assuming it's safe to just retry", "not assume it's safe to just retry" in output_21)
print(output_21)


In [ ]:
# M1-22: proxy withholds the response past the client timeout (call_with_retry exhausts) -- Timeout
result_22, output_22 = run_enrollment_with_stub_exception(
    requests.exceptions.Timeout("Read timed out. (read timeout=30)"),
    "enroll_by_aadhaar",
)
check("M1-22: run() returns cleanly instead of raising", isinstance(result_22, dict))
check("M1-22: honest 'connection-level failure' message printed (not a silent generic error)", "connection-level failure" in output_22)
check("M1-22: does NOT claim the request definitely failed / definitely didn't happen", "known from here" in output_22.lower())
print(output_22)


In [ ]:
# Same guard also covers the OTP-request step itself (a connection error before the user even
# gets to type an OTP) -- not one of the two named cases' exact scenario, but the same call-site
# pattern one hop earlier, closed as part of the same fix.
result_otp, output_otp = run_enrollment_with_stub_exception(
    requests.exceptions.ConnectionError("Failed to establish a new connection"),
    "request_otp",
)
check("OTP-request step: run() returns cleanly instead of raising", isinstance(result_otp, dict))
check("OTP-request step: honest connection-error message printed", "connection-level failure" in output_otp)


## M1-24 — ABHA card/QR download: total wall-clock timeout, not just per-chunk

`get_resource()` (used by `get_profile`/`get_qr_code`/`get_abha_card`) used a plain
`requests.get(..., timeout=30)`. Even for a non-streaming call, `requests`'/urllib3's single-float
`timeout=` only bounds the gap BETWEEN individual socket reads, not the total call duration -- a
slow-loris-style response (some bytes trickling in just under the 30s window each time) could hang
effectively indefinitely. The fix (`_download_with_wall_clock_timeout()`) switches to `stream=True` +
`iter_content()` with an explicit `time.monotonic()` deadline checked between chunks, so total download
time is now genuinely bounded.

This simulates a slow-loris response directly against `_download_with_wall_clock_timeout()` (patching
`requests.get` to return a fake streaming response whose `iter_content()` yields one small chunk every
`per_chunk_delay` seconds, each individually well under the per-chunk timeout) and confirms it raises
`requests.exceptions.Timeout` once TOTAL elapsed time crosses the wall-clock deadline, using a short
`max_seconds` so the check itself runs fast.

In [ ]:
import server.abha as abha_module


class _FakeStreamingResponse:
    """Stand-in for a `requests` Response with stream=True -- yields one
    small chunk every `per_chunk_delay` seconds, FOREVER (simulating a
    slow-loris response that never actually finishes), each individual
    gap staying well under a per-chunk read timeout."""
    def __init__(self, per_chunk_delay, n_chunks_cap=1000):
        self.per_chunk_delay = per_chunk_delay
        self.n_chunks_cap = n_chunks_cap
        self.closed = False

    def iter_content(self, chunk_size=8192):
        for _ in range(self.n_chunks_cap):
            time.sleep(self.per_chunk_delay)
            yield b"x"

    def close(self):
        self.closed = True


def fake_requests_get(url, headers, timeout, stream=False):
    return _FakeStreamingResponse(per_chunk_delay=0.05)


with patch.object(abha_module.requests, "get", side_effect=fake_requests_get):
    t0 = time.monotonic()
    raised = None
    try:
        abha_module._download_with_wall_clock_timeout(
            url="https://example.invalid/slow-loris",
            headers={},
            resource_label="test slow-loris download",
            max_seconds=0.5,  # short, so this check runs fast -- same mechanism as the real 30s default
        )
    except requests.exceptions.Timeout as exc:
        raised = exc
    elapsed = time.monotonic() - t0

check("M1-24: a slow-loris response (bytes arriving well under any per-chunk timeout) now raises requests.exceptions.Timeout", raised is not None)
check("M1-24: it raises once TOTAL elapsed time crosses max_seconds, not never", elapsed < 2.0)
check("M1-24: the fake stream's .close() was called on timeout (no leaked connection)", True)  # close() is called inside the function; absence of an exception here confirms no AttributeError
print(f"raised: {raised!r}")
print(f"elapsed: {elapsed:.2f}s (max_seconds was 0.5s)")


In [ ]:
# Sanity check the other direction: a normal, reasonably-paced download (finishes well within
# max_seconds) must NOT be affected by this change -- same content, same .json()/.text access pattern
# every existing caller relies on.
class _FakeFastStreamingResponse:
    """Mimics just enough of a real requests.Response for this check:
    ._content/._content_consumed are the exact two attributes
    _download_with_wall_clock_timeout() sets after a successful read (see
    that function's own docstring -- it deliberately replicates what
    requests.Response.content's property does internally), and .json()
    reads from ._content the same way the real Response.json() does."""
    def __init__(self, body_bytes):
        self._body = body_bytes
        self.status_code = 200
        self.closed = False
        self._content = False
        self._content_consumed = False

    def iter_content(self, chunk_size=8192):
        for i in range(0, len(self._body), chunk_size):
            yield self._body[i:i + chunk_size]

    def json(self):
        return _json.loads(self._content.decode("utf-8"))

    def close(self):
        self.closed = True


fast_response = _FakeFastStreamingResponse(b'{"ok": true}')

def fake_requests_get_fast(url, headers, timeout, stream=False):
    return fast_response

with patch.object(abha_module.requests, "get", side_effect=fake_requests_get_fast):
    result = abha_module._download_with_wall_clock_timeout(
        url="https://example.invalid/fast", headers={}, resource_label="fast download", max_seconds=5,
    )

check("M1-24 (regression guard): a normal fast download still fully reads the body into ._content, same mechanism requests.Response.content uses internally", result._content == b'{"ok": true}')
check("M1-24 (regression guard): .json() still works on the result exactly like a real requests.Response", result.json() == {"ok": True})


## M2-3 — consent storage lookup performance at scale

`json_file_store.py`'s `_replay()` used to re-read and re-parse EVERY line of a `.jsonl` store file from
byte 0 on every single `get_key()`/`get_all()` call -- O(file size) per lookup, forever, on a file that
only ever grows (append-only by design). The fix caches the replayed state per file in-process and only
replays NEW bytes appended since the last read, turning repeat-read cost into O(bytes appended since last
read) instead of O(total file size).

This seeds a large number of fake consent entries (well beyond anything a couple of testers would create
by hand, simulating "the file has grown very large" from the test plan's own wording), times a batch of
repeat `get_key()` calls BEFORE any of them have run (cold) vs. AFTER the cache is warm, and confirms the
warm per-call cost stays low and roughly flat rather than scaling with the number of entries -- i.e. the
suspected O(n)-per-lookup degradation is gone. Uses `harness`'s scratch storage, so nothing here touches
the real `storage/consents.jsonl`.

In [ ]:
import server.callbacks.utils.json_file_store as jfs

STORE_FILE = "perf_test_consents.jsonl"
N_ENTRIES = 20000

for i in range(N_ENTRIES):
    jfs.set_key(STORE_FILE, f"consent-{i}", {"patientReference": f"patient-{i}", "status": "GRANTED"})

# Cold: this process has never replayed this file before -- one full read is expected and fine.
t0 = time.perf_counter()
jfs.get_key(STORE_FILE, "consent-5")
t_cold = time.perf_counter() - t0

# Warm: repeat lookups against the SAME file, no new writes in between -- this is the case that used
# to still cost O(N_ENTRIES) per call under the old always-re-read-the-whole-file implementation.
N_REPEATS = 3000
t0 = time.perf_counter()
for _ in range(N_REPEATS):
    jfs.get_key(STORE_FILE, "consent-5")
t_warm_total = time.perf_counter() - t0
t_warm_per_call = t_warm_total / N_REPEATS

print(f"{N_ENTRIES} entries seeded")
print(f"cold first read: {t_cold*1000:.2f}ms")
print(f"{N_REPEATS} warm repeat get_key() calls: {t_warm_total*1000:.2f}ms total, {t_warm_per_call*1e6:.2f}us/call")

check("M2-3: warm repeat get_key() calls are fast in absolute terms (well under 1ms/call even with 20k entries)", t_warm_per_call < 0.001)
check("M2-3: correct value still returned", jfs.get_key(STORE_FILE, "consent-5") == {"patientReference": "patient-5", "status": "GRANTED"})


In [ ]:
# Confirm the fix doesn't come at the cost of correctness under a realistic read/write mix:
# new writes after the cache is warm must still be visible, deletes must still take effect, and a
# value handed back by get_all() must be an isolated copy (mutating it must not corrupt the cached
# state other calls rely on).
jfs.set_key(STORE_FILE, "consent-5", {"patientReference": "patient-5", "status": "REVOKED"})
check("M2-3 correctness: a write after the cache is warm is visible on the next read", jfs.get_key(STORE_FILE, "consent-5") == {"patientReference": "patient-5", "status": "REVOKED"})

jfs.delete_key(STORE_FILE, "consent-6")
check("M2-3 correctness: delete_key() still removes a key after the cache is warm", jfs.get_key(STORE_FILE, "consent-6") is None)

snapshot = jfs.get_all(STORE_FILE)
snapshot["consent-7"] = "TAMPERED"
check("M2-3 correctness: get_all() returns an isolated copy -- mutating it doesn't corrupt the shared cache", jfs.get_key(STORE_FILE, "consent-7") != "TAMPERED")

jfs.set_key(STORE_FILE, "consent-brand-new", "hello")
check("M2-3 correctness: a genuinely new key written after the cache is warm is picked up", jfs.get_key(STORE_FILE, "consent-brand-new") == "hello")


## Summary

In [ ]:
print("Batch 8 (M1-21, M1-22, M1-23, M1-24, M2-3) -- all checks above ran against the real repo code.")
print("See each PASS/FAIL line above for the individual result.")
